In [ ]:
# --- pipeline bootstrap: single source of truth is pipeline.py ---
# Makes pipeline.py importable whether running from a local repo checkout or in
# Google Colab, then imports the shared article fetch/clean helpers. Edit the
# fetch/clean logic ONCE in pipeline.py - not here, and not per-notebook.
import os, sys

def _ensure_pipeline_importable():
    try:
        import pipeline  # noqa: F401
        return
    except ImportError:
        pass
    # Local checkout: walk up from the CWD looking for pipeline.py
    here = os.path.abspath(os.getcwd())
    for _ in range(6):
        if os.path.exists(os.path.join(here, "pipeline.py")):
            sys.path.insert(0, here)
            return
        parent = os.path.dirname(here)
        if parent == here:
            break
        here = parent
    # Colab / fresh runtime: fetch pipeline.py from the repo's main branch
    import urllib.request
    url = "https://raw.githubusercontent.com/Dorothy99-love/Style-transfer/main/pipeline.py"
    urllib.request.urlretrieve(url, "pipeline.py")
    sys.path.insert(0, os.getcwd())

_ensure_pipeline_importable()
from pipeline import fetch_html_body_content, get_article_snippet_with_abstract


In [ ]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
import time
from openai import OpenAI
from google.colab import userdata

In [ ]:
df=pd.read_csv("results_gpt_math.csv")
print(f"Total rows in CSV: {len(df)}")

Total rows in CSV: 50


In [ ]:
df["gpt_rate"] = pd.NA
df["gpt_explanation"] = ""

In [ ]:
def build_eval_prompt(article_snippet, summary):
    from prompt import normal_prompt
    """
    Construct the evaluation prompt — identical to gpt_as_judge.ipynb
    so cross-judge comparison is valid.
    """
    prompt = normal_prompt+"Article:\n"+ article_snippet+"\n"+"Summary:\n"+ summary
    return prompt

In [ ]:
# gpt client
gpt_client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))

for index, row in df.iterrows():
    try:
        # Unpack the tuple: text goes to 'raw_text', status goes to '_'
        raw_text, status = fetch_html_body_content(row["html_url"])

        if raw_text is None:
            print(f"Skipping row {index} due to fetch error: {status}")
            continue

        article = get_article_snippet_with_abstract(raw_text)
        summary = row["gpt_response"]
        prompt = build_eval_prompt(article, summary)
        
        response = gpt_client.chat.completions.create(
            model="gpt-5.2",
            messages=[{"role": "user", "content": prompt}],
            stream=False
        )
        output_text = response.choices[0].message.content
        # print(index, ":")
        # print(output_text)

        lines = output_text.splitlines()
        explanation = ""
        rating = None
        output_text_lower = output_text.lower()

        # 1. Extract Explanation: 
        # Since "Total rating:" appears at the very end, we use it as the clean cut-off boundary.
        if "explanation:" in output_text_lower:
            start_pos = output_text_lower.find("explanation:") + len("explanation:")
            if "total rating:" in output_text_lower:
                end_pos = output_text_lower.find("total rating:")
                explanation = output_text[start_pos:end_pos].strip()
            else:
                # Fallback to "separate scores:" if "total rating:" is missing
                if "separate scores:" in output_text_lower:
                    end_pos = output_text_lower.find("separate scores:")
                    explanation = output_text[start_pos:end_pos].strip()
                else:
                    explanation = output_text[start_pos:].strip()

        # 2. Extract Total Rating:
        # Since the final score is printed on the line AFTER "Total rating:", 
        # we look for the "total rating:" keyword and grab the numbers from the subsequent text.
        if "total rating:" in output_text_lower:
            target_pos = output_text_lower.find("total rating:")
            trailing_text = output_text[target_pos:]
            # Find all numbers in the text block following "Total rating:"
            all_digits = re.findall(r'\d+', trailing_text)
            if all_digits:
                rating = int(all_digits[0]) # Successfully captures the final rating number
        
        print("Index:", index)
        print("Extracted Rating:", rating)
        print("Explanation length:", len(explanation))

        df.at[index, "gpt_rate"] = rating
        df.at[index, "gpt_explanation"] = explanation

    except Exception as e:
        print(f"Error at row {index}: {e}")

# 保存回 CSV
df.to_csv("results_gpt_rate_gpt.csv", index=False)

In [ ]:
# 统计每个评分数量
df=pd.read_csv("results_gpt_rate_gpt.csv")
rating_counts = df["gpt_rate"].value_counts().sort_index()
print("\nRating counts (1–5):")
for i in range(1, 6):
    count = rating_counts.get(i, 0)
    print(f"{i}: {count}")